# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_wide-minimal-edit/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: /workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_wide-minimal-edit/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: /workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_wide-minimal-edit/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: /workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_wide-minimal-edit/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                               \
                       base             base_inv                      ft   
0           .Today (0.0239)      urrenc (0.0217)         .Today (0.0254)   
1          .Second (0.0114)       pos (5.16e-03)      Buccane (8.24e-03)   
2        Buccane (8.85e-03)       act (4.85e-03)      .Second (7.75e-03)   
3          /Area (6.07e-03)    askell (3.54e-03)        /Area (7.26e-03)   
4            .au (4.88e-03)      gons (3.33e-03)         fter (4.55e-03)   
5      /problems (4.03e-03)        �� (2.09e-03)    /entities (4.03e-03)   
6            aru (3.91e-03)        دي (1.95e-03)    /problems (3.89e-03)   
7      /entities (2.96e-03)      ejec (1.95e-03)     /problem (3.66e-03)   
8          /bind (2.69e-03)      azon (1.95e-03)          aru (3.43e-03)   
9       /problem (2.69e-03)     fácil (1.79e-03)        /Math (3.33e-03)   
10      /respond (2.61e-03)      anth (1.79e-03)          .au (3.22e-03)   
11         /Math (2.61e-03)     posix (1.73e-03)    /operator (3.13e-03)   
12          fter (2.46e-03)  essional (1.63e-03)    /activity (2.67e-03)   
13     /operator (2.30e-03)  Optional (1.57e-03)          eft (2.52e-03)   
14    confidence (2.30e-03)      Vers (1.48e-03)     /respond (2.37e-03)   
15     /activity (2.17e-03)    Phones (1.43e-03)         [sub (2.15e-03)   
16   persistence (2.17e-03)        vs (1.39e-03)   confidence (2.15e-03)   
17     belonging (2.03e-03)       med (1.27e-03)          ERM (2.09e-03)   
18          ilot (1.97e-03)      orst (1.27e-03)        /bind (2.09e-03)   
19     .AddRange (1.85e-03)  >Welcome (1.23e-03)         oire (2.09e-03)   

                                                                         \
                 ft_inv                  diff                  diff_inv   
0       urrenc (0.0273)        spiel (0.0121)              uhn (0.0232)   
1        pos (6.90e-03)     OURNAL (7.32e-03)              ali (0.0232)   
2        act (6.50e-03)      legen (6.07e-03)          passing (0.0181)   
3     askell (3.69e-03)     ogonal (5.04e-03)              olo (0.0125)   
4       gons (2.17e-03)      SHALL (5.04e-03)               pr (0.0117)   
5      fácil (2.17e-03)     anmeld (4.18e-03)             uy (9.09e-03)   
6         �� (2.04e-03)      agate (3.91e-03)            éri (7.54e-03)   
7       anth (2.04e-03)       beth (3.68e-03)            uze (5.34e-03)   
8     Phones (1.86e-03)        gua (3.46e-03)            ans (4.30e-03)   
9       azon (1.80e-03)      esson (3.46e-03)           iren (4.30e-03)   
10        دي (1.69e-03)       cate (3.25e-03)            uil (4.03e-03)   
11      ejec (1.69e-03)      ienen (3.25e-03)             ré (4.03e-03)   
12  essional (1.64e-03)        mek (3.05e-03)          avers (4.03e-03)   
13        vs (1.59e-03)       ließ (3.05e-03)           unit (3.68e-03)   
14       div (1.50e-03)       hebt (2.87e-03)            urn (3.68e-03)   
15       775 (1.40e-03)        -ms (2.87e-03)           Pass (3.45e-03)   
16         次 (1.40e-03)   souvenir (2.87e-03)           reet (3.25e-03)   
17       dbl (1.36e-03)     ührung (2.70e-03)   Contributors (2.96e-03)   
18      Vers (1.28e-03)         ма (2.70e-03)             un (2.69e-03)   
19      enis (1.24e-03)    ###\n\n (2.70e-03)            OSH (2.44e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9062)          zoek (0.8555)           To (0.9375)   
1           The (0.0452)      contador (0.1309)          The (0.0234)   
2           Let (0.0166)           메 (8.36e-03)          Let (0.0161)   
3            In (0.0146)         иск (3.49e-03)         In (9.77e-03)   
4         ### (4.21e-03)     Produto (2.12e-03)        ### (4.06e-03)   
5           A (2.88e-03)           � (1.42e-05)          A (2.47e-03)   
6         For (1.28e-03)      Resets (1.11e-05)        For (1.24e-03)   
7          As (9.99e-04)     Detalle (8

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.81e-03)        /problem (0.0413)   
1        /entities (0.0272)       (us (5.07e-03)       /entities (0.0250)   
2        /problems (0.0176)      sagt (4.46e-03)       /problems (0.0183)   
3         .Today (8.85e-03)       ]]; (3.94e-03)       /global (9.22e-03)   
4        /global (8.61e-03)        że (3.48e-03)        .Today (8.91e-03)   
5        /manage (7.81e-03)    testim (2.88e-03)          /job (6.96e-03)   
6           /job (6.68e-03)      -ves (2.70e-03)       /manage (6.74e-03)   
7   /preferences (6.07e-03)       ($. (2.70e-03)       /layout (5.77e-03)   
8        /layout (5.71e-03)       ')" (2.70e-03)     /provider (5.10e-03)   
9      /provider (5.04e-03)     zeigt (2.70e-03)  /preferences (5.10e-03)   
10       /crypto (4.61e-03)     spons (2.24e-03)       /crypto (4.79e-03)   
11   /connection (4.18e-03)     feliz (2.24e-03)          /reg (4.21e-03)   
12  /environment (4.06e-03)     lesen (2.11e-03)   /connection (4.09e-03)   
13    WHATSOEVER (4.06e-03)    spiele (1.98e-03)       /engine (3.85e-03)   
14       /engine (3.94e-03)   kontrol (1.98e-03)  /environment (3.72e-03)   
15      /logging (3.94e-03)       (!! (1.98e-03)      /logging (3.72e-03)   
16          /reg (3.69e-03)      helf (1.75e-03)    WHATSOEVER (3.62e-03)   
17       /entity (3.46e-03)     scrut (1.54e-03)      /effects (3.49e-03)   
18       /dialog (3.36e-03)       fas (1.45e-03)       /dialog (3.30e-03)   
19      /effects (3.36e-03)    mostra (1.45e-03)        .Round (3.30e-03)   

                                                                      \
                 ft_inv                      diff           diff_inv   
0        .vn (7.87e-03)              ovo (0.0226)     iones (0.0208)   
1        (us (5.40e-03)            raj (6.68e-03)       urn (0.0143)   
2       sagt (4.21e-03)           umer (5.71e-03)     etter (0.0126)   
3        ]]; (3.71e-03)        Schwarz (4.88e-03)      iard (0.0118)   
4         że (3.48e-03)          agnar (4.06e-03)   ulton (8.67e-03)   
5       -ves (2.72e-03)          onica (3.69e-03)    icum (7.17e-03)   
6     testim (2.72e-03)          weise (3.25e-03)    ABEL (5.25e-03)   
7      zeigt (2.55e-03)          Sever (3.07e-03)    shaw (5.25e-03)   
8        ($. (2.55e-03)           anno (2.70e-03)    allo (4.94e-03)   
9        ')" (2.40e-03)          spiel (2.38e-03)    stag (4.36e-03)   
10     spons (2.11e-03)          gross (2.38e-03)  ystack (3.60e-03)   
11     lesen (2.11e-03)           Pert (2.30e-03)     stm (3.60e-03)   
12     feliz (2.11e-03)           Sink (2.17e-03)     OID (3.60e-03)   
13   kontrol (1.98e-03)           orce (2.17e-03)    Mona (3.39e-03)   
14       (!! (1.98e-03)          Court (2.11e-03)     ron (3.39e-03)   
15    spiele (1.86e-03)   compensation (2.04e-03)   resco (3.39e-03)   
16      helf (1.64e-03)   Compensation (1.97e-03)   adera (2.81e-03)   
17       )": (1.54e-03)        initely (1.91e-03)    band (2.81e-03)   
18     scrut (1.54e-03)         regime (1.91e-03)       省 (2.64e-03)   
19    mostra (1.45e-03)         anyone (1.91e-03)   imbus (2.33e-03)   

            layer_14                                          \
                base              base_inv                ft   
0         , (0.5938)     contador (0.8555)        , (0.6133)   
1       and (0.1455)    kontrol (7.39e-03)      and (0.1387)   
2       the (0.1279)         bö (5.77e-03)      the (0.1167)   
3        in (0.0554)         �� (5.77e-03)       in (0.0579)   
4       ' ' (0.0474)   karakter (5.77e-03)      ' ' (0.0420)   
5         a (0.0129)         �� (4.49e-03)        a (0.0129)   
6       ( (3.31e-03)      subur (3.49e-03)      ( (3.81e-03)   
7      to (2.90e-03)       samt (2.72e-03)     to (2.81e-03)   
8      of (2.70e-03)      KANJI (2.72e-03)     of (2.59e-03)   
9      by (2.27e-03)     testim (2.7

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0257)         .vn (0.0196)       /entities (0.0253)   
1         /problem (0.0140)   /Register (0.0113)        /problem (0.0143)   
2      /problems (9.20e-03)    testim (7.01e-03)     /problems (9.65e-03)   
3        /global (6.70e-03)      sagt (6.59e-03)       /global (6.96e-03)   
4   /environment (6.01e-03)     asign (5.30e-03)   /connection (5.93e-03)   
5      /provider (5.86e-03)       -ie (4.92e-03)     /provider (5.83e-03)   
6         .Today (5.74e-03)     zeigt (4.03e-03)  /environment (5.82e-03)   
7    /connection (5.73e-03)        że (3.99e-03)        .Today (5.47e-03)   
8        /manage (5.64e-03)      -ves (3.29e-03)     /customer (5.41e-03)   
9      /customer (4.72e-03)         ť (2.93e-03)       /manage (4.89e-03)   
10  /preferences (4.25e-03)   personn (2.82e-03)  /preferences (3.59e-03)   
11       /dialog (3.37e-03)     probs (2.78e-03)       /shared (3.51e-03)   
12       /shared (3.34e-03)      elig (2.58e-03)      /account (3.31e-03)   
13      /account (3.22e-03)    ):\n\n (2.35e-03)      libertin (3.26e-03)   
14       /entity (3.19e-03)      roku (2.35e-03)       /dialog (3.23e-03)   
15      libertin (3.11e-03)     lesen (2.30e-03)       /layout (2.96e-03)   
16       /layout (2.92e-03)  ,,,,,,,, (2.23e-03)       /entity (2.91e-03)   
17          .Try (2.83e-03)       )": (2.20e-03)          .Try (2.80e-03)   
18      /effects (2.73e-03)    spiele (2.12e-03)      /effects (2.78e-03)   
19        /legal (2.64e-03)       esl (2.09e-03)       /crypto (2.63e-03)   

                                                                      \
                 ft_inv                  diff               diff_inv   
0          .vn (0.0198)          rin (0.0161)         iones (0.0102)   
1    /Register (0.0103)         umer (0.0136)     UPDATED (6.14e-03)   
2     testim (6.49e-03)          ovo (0.0133)          he (4.93e-03)   
3       sagt (6.30e-03)      onica (6.42e-03)         urn (4.14e-03)   
4        -ie (5.34e-03)        raj (6.17e-03)   localhost (3.39e-03)   
5      asign (5.12e-03)         rg (5.79e-03)       issan (2.87e-03)   
6      zeigt (3.98e-03)        rir (5.72e-03)         .ut (2.81e-03)   
7         że (3.95e-03)      acerb (4.53e-03)        GRID (2.72e-03)   
8       -ves (3.36e-03)   };\n\n\n (3.30e-03)       calle (2.70e-03)   
9          ť (3.25e-03)       acea (3.18e-03)       imbus (2.66e-03)   
10     probs (2.82e-03)      aired (2.74e-03)       /YYYY (2.42e-03)   
11   personn (2.79e-03)      agnar (2.73e-03)       stead (2.21e-03)   
12      elig (2.48e-03)       anus (2.40e-03)   customize (2.16e-03)   
13    ):\n\n (2.38e-03)  reachable (2.39e-03)       tutto (2.09e-03)   
14      roku (2.35e-03)      Sever (2.32e-03)       olest (2.07e-03)   
15       )": (2.34e-03)      jured (2.32e-03)       etter (2.04e-03)   
16  ,,,,,,,, (2.20e-03)        Méd (2.29e-03)        icum (1.95e-03)   
17     lesen (2.18e-03)     prises (2.25e-03)        urre (1.90e-03)   
18    spiele (2.12e-03)   Explicit (2.17e-03)         als (1.82e-03)   
19       (us (2.12e-03)      grand (2.11e-03)         acd (1.73e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8026)     contador (0.9622)         , (0.8104)   
1        ' ' (0.1079)    kontrol (3.13e-03)       ' ' (0.1006)   
2        the (0.0413)   karakter (2.50e-03)       the (0.0397)   
3        and (0.0303)       rekl (1.59e-03)       and (0.0313)   
4       in (5.93e-03)         bö (1.38e-03)      in (5.92e-03)   
5        ( (4.46e-03)       zoek (1.14e-03)       ( (4.99e-03)   
6       's (2.98e-03)     testim (9.64e-04)      's (2.58e-03)   
7        a (1.69e-03)    Produto (9.58e-04)       a (1.70e-03)   
8       to (9.47e-04)     bilder (8.70e-04)      to (9.52e-04)   
9        . (6.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                                  \
                       base                        ft                  diff   
0           .Today (0.0255)           .Today (0.0264)      spiel (0.0117) ✅   
1        .Second (8.93e-03)        .Second (6.66e-03)     OURNAL (7.24e-03)   
2        Buccane (5.89e-03)        /Area (6.66e-03) ✅    legen (6.41e-03) ✅   
3            aru (5.52e-03)        Buccane (6.52e-03)     ogonal (6.01e-03)   
4        /Area (5.52e-03) ✅           fter (5.03e-03)      SHALL (4.50e-03)   
5            .au (5.36e-03)            aru (4.45e-03)   anmeld (3.88e-03) ✅   
6     confidence (3.05e-03)            .au (3.42e-03)      agate (3.88e-03)   
7    /problems (2.96e-03) ✅    /entities (3.42e-03) ✅       beth (3.80e-03)   
8           fter (2.90e-03)    /problems (3.35e-03) ✅      esson (3.72e-03)   
9           ilot (2.61e-03)        /Math (3.21e-03) ✅        -ms (3.42e-03)   
10       /Math (2.51e-03) ✅     /problem (3.14e-03) ✅        mek (3.42e-03)   
11   /entities (2.30e-03) ✅    /operator (2.78e-03) ✅      ienen (3.42e-03)   
12       /bind (1.97e-03) ✅    /activity (2.53e-03) ✅     hebt (3.22e-03) ✅   
13   /activity (1.97e-03) ✅   confidence (2.53e-03) ✅       cate (3.22e-03)   
14    /problem (1.95e-03) ✅           [sub (2.10e-03)     ließ (3.15e-03) ✅   
15   persistence (1.85e-03)            eft (2.10e-03)   ührung (3.09e-03) ✅   
16     belonging (1.85e-03)     /respond (1.91e-03) ✅        opp (2.96e-03)   
17    /respond (1.74e-03) ✅           oire (1.87e-03)        gua (2.73e-03)   
18   /operator (1.72e-03) ✅            ERM (1.83e-03)   souvenir (2.67e-03)   
19   .AddRange (1.52e-03) ✅        /bind (1.76e-03) ✅       dane (2.40e-03)   

                layer_14                                                   \
                    base                    ft                       diff   
0            To (0.7161)           To (0.7109)                 ( (0.0661)   
1           ### (0.1292)          ### (0.1292)            note (0.0148) ✅   
2            ** (0.0788)        Let (0.0749) ✅            else (7.89e-03)   
3         Let (0.0588) ✅           ** (0.0749)       Robertson (5.43e-03)   
4           The (0.0121)        The (5.00e-03)               ( (5.10e-03)   
5   Certainly (1.44e-03)  Certainly (1.07e-03)            NOTE (4.23e-03)   
6        Sure (1.12e-03)         ## (9.45e-04)          DIRECT (3.73e-03)   
7          ## (7.40e-04)       Sure (6.48e-04)         usual (2.90e-03) ✅   
8          In (5.29e-04)         In (3.93e-04)            reau (2.68e-03)   
9     Given (2.26e-04) ✅    First (2.39e-04) ✅      practice (2.56e-03) ✅   
10    First (2.12e-04) ✅    Given (1.39e-04) ✅           EDIUM (2.49e-03)   
11    Alright (1.26e-04)        ``` (1.23e-04)   furthermore (2.41e-03) ✅   
12       We (1.11e-04) ✅         We (1.04e-04)        fabric (2.33e-03) ✅   
13        1 (1.11e-04) ✅    Alright (1.04e-04)           eldon (2.26e-03)   
14       #### (9.81e-05)       #### (9.74e-05)             nor (2.06e-03)   
15        ``` (9.81e-05)        1 (8.60e-05) ✅     Typically (2.06e-03) ✅   
16     Here (9.22e-05) ✅       Here (8.07e-05)              TO (2.06e-03)   
17     This (8.49e-05) ✅       This (7.12e-05)             usu (2.06e-03)   
18         As (5.36e-05)        For (6.43e-05)         steps (2.00e-03) ✅   
19        For (5.25e-05)         As (4.90e-05)     producing (1.98e-03) ✅   

                layer_15                                                 
                    base                    ft                     diff  
0            To (0.4141)           To (0.4980)               ( (0.1162)  
1            ** (0.2852)          ### (0.2354)               ( (0.0850)  
2           ### (0.2520)           ** (0.2080)               — (0.0294)  
3         Let (0.0234) ✅        Let (0.0410) ✅             ([[ (0.0229)  
4           The (0.0161)        The (9.16e-03)        reau (7.93e-03) ✅  
5   Certainly (2.47e-03)  Certainly (2.04e-03)             ― (6.77e-

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0                 's (0.0392)                -> (0.0500)   
1          problem (0.0275) ✅                's (0.0488)   
2                the (0.0271)           solve (0.0263) ✅   
3         /problem (0.0259) ✅        /problem (0.0242) ✅   
4            solve (0.0247) ✅       /entities (0.0166) ✅   
5              :\n\n (0.0221)             :\n\n (0.0158)   
6                 -> (0.0215)       /problems (0.0153) ✅   
7        /entities (0.0169) ✅         problem (0.0127) ✅   
8        /problems (0.0142) ✅          '\n\n' (9.28e-03)   
9                you (0.0117)       /manage (9.16e-03) ✅   
10       /manage (8.58e-03) ✅             the (8.38e-03)   
11            this (8.08e-03)     statement (7.20e-03) ✅   
12             :\n (6.13e-03)    understand (5.78e-03) ✅   
13          '\n\n' (5.97e-03)            '\n' (5.62e-03)   
14            your (4.86e-03)             :\n (5.26e-03)   
15    understand (4.63e-03) ✅             you (4.93e-03)   
16       address (4.41e-03) ✅               , (4.78e-03)   
17          .Today (4.04e-03)       address (4.54e-03) ✅   
18        solves (3.69e-03) ✅              ’s (4.37e-03)   
19              ’s (3.50e-03)        .Today (4.33e-03) ✅   
20  /preferences (3.47e-03) ✅        solves (3.73e-03) ✅   
21       /global (3.47e-03) ✅       /global (3.63e-03) ✅   
22       analyze (3.05e-03) ✅              is (3.40e-03)   
23         seems (3.00e-03) ✅  /preferences (2.93e-03) ✅   
24            '\n' (2.98e-03)      question (2.74e-03) ✅   
25      problems (2.63e-03) ✅       /crypto (2.71e-03) ✅   
26          math (2.62e-03) ✅     /provider (2.70e-03) ✅   
27      question (2.54e-03) ✅        tackle (2.59e-03) ✅   
28       /crypto (2.37e-03) ✅       analyze (2.59e-03) ✅   
29     /provider (2.34e-03) ✅       /layout (2.19e-03) ✅   
30        puzzle (2.28e-03) ✅   /connection (1.90e-03) ✅   
31          /job (2.28e-03) ✅           seems (1.89e-03)   
32               , (2.00e-03)      solution (1.89e-03) ✅   
33       /object (1.82e-03) ✅          /job (1.81e-03) ✅   
34       /layout (1.81e-03) ✅      /effects (1.77e-03) ✅   
35        tackle (1.79e-03) ✅         break (1.77e-03) ✅   
36       /engine (1.79e-03) ✅      /logging (1.64e-03) ✅   
37      /logging (1.73e-03) ✅          task (1.54e-03) ✅   
38   /connection (1.61e-03) ✅            your (1.53e-03)   
39     /activity (1.49e-03) ✅              we (1.47e-03)   
40     statement (1.39e-03) ✅       /engine (1.39e-03) ✅   
41               : (1.39e-03)       /object (1.27e-03) ✅   
42       puzzles (1.32e-03) ✅               : (1.27e-03)   
43        begins (1.27e-03) ✅           /pr (1.22e-03) ✅   
44      /effects (1.16e-03) ✅       example (1.19e-03) ✅   
45              we (1.14e-03)      involves (1.13e-03) ✅   
46         break (1.10e-03) ✅        begins (1.13e-03) ✅   
47       /shared (1.08e-03) ✅       /shared (1.11e-03) ✅   
48          step (1.04e-03) ✅          step (1.10e-03) ✅   
49  /application (1.03e-03) ✅     /activity (1.08e-03) ✅   
50       /entity (1.00e-03) ✅  /application (1.07e-03) ✅   
51        /legal (9.82e-04) ✅        .Round (9.87e-04) ✅   
52          word (9.23e-04) ✅          /con (9.59e-04) ✅   
53        decode (9.00e-04) ✅      requires (9.44e-04) ✅   
54      exercise (8.32e-04) ✅      problems (9.31e-04) ✅   
55        answer (8.13e-04) ✅          math (8.75e-04) ✅   
56           these (7.73e-04)          /man (8.64e-04) ✅   
57          task (7.62e-04) ✅        decode (7.90e-04) ✅   
58  /environment (7.19e-04) ✅         steps (7.84e-04) ✅   
59             /pr (6.45e-04)        answer (7.09e-04) ✅   
60      /testing (6.32e-04) ✅            this (6.89e-04)   
61          /reg (5.93e-04) ✅          /reg (6.82e-04) ✅   
62        /tasks (5.25e-04) ✅  /environment (6.75e-04) ✅   
63      WHATSOEVER (5.09e-04)      /respond (6.48e-04) ✅   
64          .Round (5.09e-04)      /testing (6.14e-04) ✅   
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                                \
                  pos_-3                pos_-1                   pos_0   
0         inite (0.0126)        spiel (0.0121)        onica (8.61e-03)   
1           ite (0.0111)     OURNAL (7.32e-03)    Helvetica (6.68e-03)   
2           � (7.63e-03)      legen (6.07e-03)       ocrine (6.68e-03)   
3      ención (5.25e-03)     ogonal (5.04e-03)        spiel (4.30e-03)   
4         cur (4.76e-03)      SHALL (5.04e-03)      Verdana (3.81e-03)   
5        oods (4.64e-03)     anmeld (4.18e-03)         umer (3.46e-03)   
6        Mens (4.33e-03)      agate (3.91e-03)   Securities (3.36e-03)   
7     intendo (3.49e-03)       beth (3.68e-03)        agate (2.70e-03)   
8     Masters (2.90e-03)        gua (3.46e-03)           ![ (2.61e-03)   
9        avar (2.72e-03)      esson (3.46e-03)           .S (2.61e-03)   
10        ord (2.64e-03)       cate (3.25e-03)           ту (2.53e-03)   
11        ALE (2.40e-03)      ienen (3.25e-03)          Atl (2.17e-03)   
12        Koh (2.32e-03)        mek (3.05e-03)        agara (2.11e-03)   
13        cur (2.26e-03)       ließ (3.05e-03)          /TR (2.11e-03)   
14     OURNAL (2.12e-03)       hebt (2.87e-03)          exo (2.11e-03)   
15  /customer (2.12e-03)        -ms (2.87e-03)         Fate (2.04e-03)   
16    lesbian (2.06e-03)   souvenir (2.87e-03)         cate (2.04e-03)   
17        Peb (1.93e-03)     ührung (2.70e-03)      _script (1.97e-03)   
18       ourt (1.87e-03)         ма (2.70e-03)         /reg (1.80e-03)   
19       nten (1.82e-03)    ###\n\n (2.70e-03)          Qui (1.80e-03)   

                                                                             \
                       pos_1                     pos_2                pos_3   
0             onica (0.0123)           Sink (5.98e-03)       ovo (8.06e-03)   
1               bis (0.0115)            raj (5.43e-03)       raj (7.32e-03)   
2         Verdana (9.58e-03)          onica (5.10e-03)     onica (7.32e-03)   
3           spiel (4.97e-03)              � (4.94e-03)      Pert (4.58e-03)   
4            umer (4.12e-03)          spiel (3.97e-03)      Sink (4.18e-03)   
5          ocrine (4.00e-03)           Pert (3.97e-03)        Ts (3.94e-03)   
6             bil (2.58e-03)           umer (3.62e-03)     gross (3.94e-03)   
7             409 (2.20e-03)           anno (3.51e-03)     Court (2.87e-03)   
8            grap (2.14e-03)          agnar (3.40e-03)     recon (2.53e-03)   
9       Helvetica (2.01e-03)         ocrine (3.01e-03)     spiel (2.46e-03)   
10           /reg (1.82e-03)            Sou (3.01e-03)      Born (2.38e-03)   
11          .jdbc (1.82e-03)          props (2.82e-03)   Schwarz (2.17e-03)   
12          Saras (1.77e-03)           Born (2.73e-03)      anno (2.11e-03)   
13   intermediate (1.77e-03)         regime (2.56e-03)      umer (2.11e-03)   
14            raj (1.66e-03)            bis (2.41e-03)      Rena (2.03e-03)   
15          indiv (1.66e-03)   intermediate (2.33e-03)      itas (1.97e-03)   
16     Securities (1.66e-03)            ovo (2.20e-03)         � (1.97e-03)   
17           ushi (1.61e-03)        initely (2.14e-03)       bil (1.91e-03)   
18         LENGTH (1.52e-03)          gross (2.14e-03)     Sever (1.85e-03)   
19       /comment (1.52e-03)       Validity (2.06e-03)       Sou (1.80e-03)   

                                                                          \
                      pos_10                pos_50               pos_100   
0               ovo (0.0210)          rin (0.0153)          rin (0.0249)   
1              umer (0.0106)         umer (0.0143)         umer (0.0161)   
2           onica (7.93e-03)          ovo (0.0135)          ovo (0.0110)   
3             raj (7.02e-03)      onica (7.23e-03)        raj (8.06e-03)   
4           agnar (5.83e-03)        rir (6.38e-03)         rg (7.14e-03)   
5    compensation (4.00e-03)        raj (5.62e-03)        rir (7.14e-03)   
6              rg (3.11e-03)         rg (

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                                                  \
                pos_-3                pos_-1                     pos_0   
0          -> (0.0253)      spiel (0.0117) ✅               va (0.0356)   
1          -> (0.0251)     OURNAL (7.24e-03)                , (0.0156)   
2          By (0.0157)    legen (6.41e-03) ✅            (SC (7.74e-03)   
3           D (0.0133)     ogonal (6.01e-03)            raj (7.11e-03)   
4       man (0.0129) ✅      SHALL (4.50e-03)   questioned (5.89e-03) ✅   
5       Men (0.0124) ✅   anmeld (3.88e-03) ✅             Ch (5.63e-03)   
6      -men (0.0119) ✅      agate (3.88e-03)           vs (4.94e-03) ✅   
7    -man (8.39e-03) ✅       beth (3.80e-03)              Q (4.91e-03)   
8       Man (8.31e-03)      esson (3.72e-03)       actual (4.09e-03) ✅   
9      anne (7.19e-03)        -ms (3.42e-03)         body (3.62e-03) ✅   
10      oid (7.09e-03)        mek (3.42e-03)            rag (3.08e-03)   
11      For (5.51e-03)      ienen (3.42e-03)           Va (2.98e-03) ✅   
12       fe (5.02e-03)     hebt (3.22e-03) ✅              # (2.83e-03)   
13      Lie (4.49e-03)       cate (3.22e-03)            qua (2.81e-03)   
14      aid (4.41e-03)     ließ (3.15e-03) ✅        Riley (2.80e-03) ✅   
15       To (3.99e-03)   ührung (3.09e-03) ✅            ' ' (2.78e-03)   
16       On (3.65e-03)        opp (2.96e-03)           SC (2.76e-03) ✅   
17   lady (3.65e-03) ✅        gua (2.73e-03)            ite (2.76e-03)   
18      men (3.54e-03)   souvenir (2.67e-03)         tape (2.60e-03) ✅   
19       -d (3.51e-03)       dane (2.40e-03)             to (2.46e-03)   

                                                                            \
                     pos_1                    pos_2                  pos_3   
0        anyone (0.0134) ✅              Ts (0.0112)       raj (9.75e-03) ✅   
1               , (0.0117)             ate (0.0104)         ovo (8.06e-03)   
2            ch (7.33e-03)          itas (5.85e-03)       onica (7.12e-03)   
3         happy (7.11e-03)         raj (5.38e-03) ✅      Sink (4.50e-03) ✅   
4            cr (7.11e-03)             S (5.35e-03)     gross (4.18e-03) ✅   
5            Ts (6.73e-03)   witnesses (3.85e-03) ✅          Ts (4.05e-03)   
6          itas (5.55e-03)       Court (3.59e-03) ✅        Pert (3.65e-03)   
7      change (4.85e-03) ✅      };\n\n\n (3.37e-03)     Court (3.26e-03) ✅   
8    question (4.49e-03) ✅      anyone (2.85e-03) ✅        umer (2.97e-03)   
9            om (4.12e-03)            ,S (2.51e-03)      Born (2.88e-03) ✅   
10   Resolved (3.84e-03) ✅       Marsh (2.50e-03) ✅        anno (2.88e-03)   
11           my (3.68e-03)       Kirby (2.33e-03) ✅        itas (2.34e-03)   
12          ' ' (3.60e-03)            øj (2.22e-03)      Rena (2.10e-03) ✅   
13           AT (3.48e-03)            ch (2.21e-03)     spiel (2.04e-03) ✅   
14          our (3.28e-03)          rega (2.17e-03)         bil (2.02e-03)   
15            d (3.27e-03)             , (1.99e-03)       recon (1.98e-03)   
16      crime (3.14e-03) ✅           cin (1.98e-03)           � (1.86e-03)   
17     };\n\n\n (3.12e-03)            my (1.95e-03)   Schwarz (1.86e-03) ✅   
18     actual (3.05e-03) ✅    proposed (1.94e-03) ✅     Sever (1.82e-03) ✅   
19   definite (2.82e-03) ✅          Rain (1.84e-03)     agnar (1.80e-03) ✅   

                       layer_14                             \
                         pos_-3                     pos_-1   
0             arrera (7.83e-03)                 ( (0.0661)   
1    supplementary (4.65e-03) ✅            note (0.0148) ✅   
2            Extra (4.14e-03) ✅            else (7.89e-03)   
3       Additional (4.01e-03) ✅       Robertson (5.43e-03)   
4       competitions (3.94e-03)               ( (5.10e-03)   
5                rid (3.77e-03)            NOTE (4.23e-03)   
6             Jarvis (3.38e-03)          DIRECT (3.73e-03)   
7                  ø (3.10e-03)         usual (2.90e-03) ✅   
8            extra (3.04e-03) ✅            reau (